In [ ]:
# =========================
# KENYA HABITAT SIMILARITY INTERPOLATION (HEADLESS)
# - Reads two Excel files
# - Builds buffered habitat polygons from points
# - Profiles known habitats
# - Searches all Kenya for environmentally similar locations
# - Saves CSV + checkpoint
# =========================

import os
import time
import random
import gc
import csv
import pickle
import math

import numpy as np
import pandas as pd
import ee

# -------------------------
# USER CONFIG
# -------------------------
PROJECT_ID = "interpolated"

FILE_1 = "/Users/rs/Downloads/positives & negatives kenya.xlsx"
FILE_2 = "/Users/rs/Downloads/Kenya.xlsx"

BUFFER_M = 300

# similarity workflow settings
SIMILARITY_THRESHOLD = 88
CHUNK_SIZE = 1.0                 # degrees
SAMPLE_PIXELS_PER_CHUNK = 250    # lower if memory issues
SCALE = 30                       # candidate sampling scale
MAX_PIXELS = 1e13

# output
BASE_DIR = os.path.join(os.getcwd(), "KENYA_HABITAT_SIMILARITY")
os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = os.path.join(BASE_DIR, "si.csv")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoint.pkl")

# -------------------------
# EE init
# -------------------------
try:
    ee.Initialize(project=PROJECT_ID)
    print("✅ Earth Engine initialized")
except Exception as e:
    raise RuntimeError(
        "Earth Engine init failed. Run `earthengine authenticate` and restart the notebook."
    ) from e

print("BASE_DIR:", BASE_DIR)
print("CSV_PATH:", CSV_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

# =========================
# 1. Excel helpers
# =========================
def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_sheet(df, label):
    df = clean_columns(df)

    lon_candidates = [c for c in df.columns if c.lower().startswith("longitude")]
    lat_candidates = [c for c in df.columns if c.lower().startswith("latitude")]
    date_candidates = [c for c in df.columns if c.lower().startswith("date")]
    site_candidates = [c for c in df.columns if c.lower().startswith("site")]

    if not lon_candidates or not lat_candidates:
        raise ValueError(f"Could not find Longitude/Latitude columns in {list(df.columns)}")

    lon_col = lon_candidates[0]
    lat_col = lat_candidates[0]
    date_col = date_candidates[0] if date_candidates else None
    site_col = site_candidates[0] if site_candidates else None

    keep_cols = [lon_col, lat_col]
    if date_col:
        keep_cols.append(date_col)
    if site_col:
        keep_cols.append(site_col)

    out = df[keep_cols].copy()

    rename_map = {
        lon_col: "Longitude",
        lat_col: "Latitude",
    }
    if date_col:
        rename_map[date_col] = "Date"
    if site_col:
        rename_map[site_col] = "site_id"

    out = out.rename(columns=rename_map)

    if "Date" not in out.columns:
        out["Date"] = pd.NaT
    if "site_id" not in out.columns:
        out["site_id"] = range(1, len(out) + 1)

    out["Longitude"] = pd.to_numeric(out["Longitude"], errors="coerce")
    out["Latitude"] = pd.to_numeric(out["Latitude"], errors="coerce")
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["site_id"] = out["site_id"].astype(str)
    out["label"] = int(label)

    out = out.dropna(subset=["Longitude", "Latitude"]).copy()
    return out[["site_id", "Longitude", "Latitude", "Date", "label"]]

def load_all_points():
    pos1 = pd.read_excel(FILE_1, sheet_name="Positives")
    neg1 = pd.read_excel(FILE_1, sheet_name="Negatives")

    pos2 = pd.read_excel(FILE_2, sheet_name="Positive Sites")
    neg2 = pd.read_excel(FILE_2, sheet_name="Negatve Sites")

    pos1 = standardize_sheet(pos1, 1)
    neg1 = standardize_sheet(neg1, 0)
    pos2 = standardize_sheet(pos2, 1)
    neg2 = standardize_sheet(neg2, 0)

    df = pd.concat([pos1, neg1, pos2, neg2], ignore_index=True)
    df = df.drop_duplicates(subset=["Longitude", "Latitude", "label"]).reset_index(drop=True)
    return df

# =========================
# 2. CSV helpers
# =========================
def initialize_csv_local(headers):
    with open(CSV_PATH, "w", newline="") as f:
        csv.writer(f).writerow(headers)

def append_rows_to_csv(rows):
    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerows(rows)

# =========================
# 3. Build EE features/buffers
# =========================
def df_to_ee_points(df):
    features = []
    for _, row in df.iterrows():
        ft = ee.Feature(
            ee.Geometry.Point([float(row["Longitude"]), float(row["Latitude"])]),
            {
                "site_id": str(row["site_id"]),
                "label": int(row["label"]),
                "date_str": row["Date"].strftime("%Y-%m-%d") if pd.notnull(row["Date"]) else ""
            }
        )
        features.append(ft)
    return ee.FeatureCollection(features)

def make_buffer_feature(row, buffer_m):
    return ee.Feature(
        ee.Geometry.Point([float(row["Longitude"]), float(row["Latitude"])]).buffer(buffer_m),
        {
            "site_id": str(row["site_id"]),
            "label": int(row["label"]),
            "date_str": row["Date"].strftime("%Y-%m-%d") if pd.notnull(row["Date"]) else ""
        }
    )

def df_to_buffer_fc(df, buffer_m):
    features = []
    for _, row in df.iterrows():
        features.append(make_buffer_feature(row, buffer_m))
    return ee.FeatureCollection(features)

# =========================
# 4. Kenya boundary
# =========================
kenya_fc = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Kenya"))
)
roi = kenya_fc.geometry()

# Bounds for chunking
roi_bounds = roi.bounds(1).coordinates().get(0).getInfo()
xs = [pt[0] for pt in roi_bounds]
ys = [pt[1] for pt in roi_bounds]
LON_MIN, LON_MAX = min(xs), max(xs)
LAT_MIN, LAT_MAX = min(ys), max(ys)

print("✅ Using Kenya boundary")
print("Bounds:", (LON_MIN, LAT_MIN, LON_MAX, LAT_MAX))

# =========================
# 5. Sentinel-2 setup
# =========================
def mask_s2_sr(image):
    scl = image.select("SCL")
    mask = (
        scl.neq(3)   # cloud shadow
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(mask)

def add_indices(img):
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("ndwi")
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("ndvi")
    mndwi = img.normalizedDifference(["B3", "B11"]).rename("mndwi")
    return img.addBands([ndwi, ndvi, mndwi])

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2023-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
    .map(mask_s2_sr)
    .select(["B2", "B3", "B4", "B8", "B11", "SCL"])
)

print("Sentinel-2 image count after masking:", s2.size().getInfo())

median_image = s2.median().select(["B2", "B3", "B4", "B8", "B11"])
indexed_image = add_indices(median_image).clip(roi)

PROFILE_BANDS = ["ndwi", "ndvi", "mndwi", "B2", "B3", "B4", "B8", "B11"]

print("Profile bands:", PROFILE_BANDS)

# =========================
# 6. Habitat profile + weights
# =========================
def create_habitat_profile(indexed_image, polygon_feature):
    geom = polygon_feature.geometry()
    stats = indexed_image.select(PROFILE_BANDS).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), None, True),
        geometry=geom,
        scale=10,
        maxPixels=MAX_PIXELS
    )
    return ee.Dictionary(stats)

def calculate_weights(reference_fc, bands, indexed_image):
    eps = ee.Number(1e-8)
    stats_dict = ee.Dictionary()

    for band in bands:
        band_stats = indexed_image.select(band).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), None, True),
            geometry=reference_fc.geometry(),
            scale=10,
            maxPixels=MAX_PIXELS
        )
        stats_dict = stats_dict.combine(band_stats)

    weights = ee.Dictionary.fromLists(
        bands,
        ee.List(bands).map(
            lambda b: ee.Number(stats_dict.get(ee.String(b).cat("_mean")))
            .abs()
            .divide(ee.Number(stats_dict.get(ee.String(b).cat("_stdDev"))).add(eps))
        )
    )

    total = ee.Number(weights.values().reduce(ee.Reducer.sum()))
    return weights.map(lambda k, v: ee.Number(v).divide(total))

# =========================
# 7. Similarity scoring
# =========================
def _score_one_profile(ft, profile, weights, indices):
    def band_score(idx):
        idx = ee.String(idx)
        x = ee.Number(ft.get(idx))
        mean = ee.Number(profile.get(idx.cat("_mean")))
        std = ee.Number(profile.get(idx.cat("_stdDev"))).max(1e-8)
        w = ee.Number(weights.get(idx))
        sim = ee.Number(1).divide(ee.Number(1).add(x.subtract(mean).abs().divide(std)))
        return sim.multiply(w)

    weighted_sum = ee.Number(ee.List(indices).map(band_score).reduce(ee.Reducer.sum()))
    total_w = ee.Number(
        ee.List(indices).map(lambda i: ee.Number(weights.get(ee.String(i)))).reduce(ee.Reducer.sum())
    )
    return weighted_sum.divide(total_w).multiply(100)

def best_match_feature(ft, profiles_list, weights, indices):
    init = ee.Dictionary({"bestSim": ee.Number(-1), "bestIdx": ee.Number(-1)})

    def step(profile_and_idx, acc):
        acc = ee.Dictionary(acc)
        profile = ee.Dictionary(ee.List(profile_and_idx).get(0))
        idx = ee.Number(ee.List(profile_and_idx).get(1))

        sim = _score_one_profile(ft, profile, weights, indices)
        bestSim = ee.Number(acc.get("bestSim"))
        better = sim.gt(bestSim)

        return ee.Dictionary({
            "bestSim": ee.Algorithms.If(better, sim, bestSim),
            "bestIdx": ee.Algorithms.If(better, idx, ee.Number(acc.get("bestIdx"))),
        })

    enumerated = profiles_list.zip(ee.List.sequence(0, profiles_list.size().subtract(1)))
    out = ee.Dictionary(enumerated.iterate(step, init))

    return ft.set({
        "similarity": ee.Number(out.get("bestSim")),
        "habitat_index": ee.Number(out.get("bestIdx"))
    })

# =========================
# 8. Checkpoint helper
# =========================
def ensure_checkpoint_has_buffers(checkpoint):
    if "buffer_features" in checkpoint and checkpoint["buffer_features"] is not None:
        if "processed_chunk_indices" in checkpoint and not isinstance(checkpoint["processed_chunk_indices"], set):
            checkpoint["processed_chunk_indices"] = set(checkpoint["processed_chunk_indices"])
        return checkpoint

    print("⚠️ Checkpoint missing buffered habitat features. Rebuilding...")
    df = load_all_points()
    checkpoint["buffer_features"] = df.to_dict("records")

    if "processed_chunk_indices" in checkpoint and not isinstance(checkpoint["processed_chunk_indices"], set):
        checkpoint["processed_chunk_indices"] = set(checkpoint["processed_chunk_indices"])

    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(checkpoint, f)

    print(f"✅ Rebuilt checkpoint with {len(df)} buffered habitats")
    return checkpoint

# =========================
# 9. Core processing
# =========================
def process_kenya_similarity_headless(
    df_points,
    similarity_threshold=88,
    chunk_size=1.0,
    sample_pixels_per_chunk=250,
    scale=30,
):
    if not os.path.exists(CSV_PATH):
        initialize_csv_local([
            "Latitude", "Longitude", "Habitat ID", "Similarity Score", "Chunk Index"
        ])

    points_fc = df_to_ee_points(df_points)
    buffered_fc = df_to_buffer_fc(df_points, BUFFER_M)

    # Build habitat profiles
    buffered_list = buffered_fc.toList(buffered_fc.size())
    n_buffers = buffered_fc.size().getInfo()
    print("Building profiles from buffered habitats:", n_buffers)

    habitat_profiles = []
    for k in range(n_buffers):
        prof = create_habitat_profile(indexed_image, ee.Feature(buffered_list.get(k)))
        habitat_profiles.append(prof)
        if (k + 1) % 10 == 0 or (k + 1) == n_buffers:
            print(f"Built profile {k+1}/{n_buffers}")

    profiles_list = ee.List(habitat_profiles)
    weights = calculate_weights(buffered_fc, PROFILE_BANDS, indexed_image)

    print("Computed weights.")

    # Chunks
    chunks = []
    for x in np.arange(LON_MIN, LON_MAX, chunk_size):
        for y in np.arange(LAT_MIN, LAT_MAX, chunk_size):
            chunks.append((
                float(x),
                float(y),
                float(min(x + chunk_size, LON_MAX)),
                float(min(y + chunk_size, LAT_MAX))
            ))

    print("Chunks:", len(chunks))

    # load checkpoint
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "rb") as f:
            checkpoint = pickle.load(f)
        checkpoint = ensure_checkpoint_has_buffers(checkpoint)
        processed = checkpoint.get("processed_chunk_indices", set())
        if not isinstance(processed, set):
            processed = set(processed)
    else:
        checkpoint = {}
        processed = set()

    remaining = [i for i in range(len(chunks)) if i not in processed]
    print(f"Chunks total={len(chunks)} | remaining={len(remaining)} | threshold={similarity_threshold}")

    for idx in remaining:
        xmin, ymin, xmax, ymax = chunks[idx]
        t0 = time.time()
        print(f"\n▶️ START chunk {idx}/{len(chunks)-1}: [{xmin:.3f},{ymin:.3f}]–[{xmax:.3f},{ymax:.3f}]")

        chunk_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

        # Skip chunks outside Kenya
        intersects = roi.intersects(chunk_geom, ee.ErrorMargin(1)).getInfo()
        if not intersects:
            processed.add(idx)
            checkpoint.update({
                "lon_min": LON_MIN,
                "lat_min": LAT_MIN,
                "lon_max": LON_MAX,
                "lat_max": LAT_MAX,
                "similarity_threshold": similarity_threshold,
                "chunk_size": chunk_size,
                "sample_pixels_per_chunk": sample_pixels_per_chunk,
                "processed_chunk_indices": processed,
                "buffer_features": df_points.to_dict("records"),
            })
            with open(CHECKPOINT_PATH, "wb") as f:
                pickle.dump(checkpoint, f)
            print("Chunk outside Kenya, skipped.")
            continue

        # sample candidate pixels
        samples = indexed_image.select(PROFILE_BANDS).sample(
            region=chunk_geom.intersection(roi, ee.ErrorMargin(1)),
            scale=scale,
            numPixels=sample_pixels_per_chunk,
            geometries=True,
            dropNulls=True
        )

        scored = samples.map(lambda f: best_match_feature(f, profiles_list, weights, ee.List(PROFILE_BANDS)))
        filtered = scored.filter(ee.Filter.gte("similarity", similarity_threshold))

        try:
            info = filtered.getInfo()
        except Exception as e:
            msg = str(e).lower()

            if ("memory limit" in msg) or ("user memory limit exceeded" in msg):
                if sample_pixels_per_chunk > 50:
                    new_n = max(50, int(sample_pixels_per_chunk * 0.5))
                    print(f"⚠️ Memory limit on chunk {idx}. Reducing samples {sample_pixels_per_chunk} -> {new_n} and retrying later...")
                    checkpoint.update({
                        "lon_min": LON_MIN,
                        "lat_min": LAT_MIN,
                        "lon_max": LON_MAX,
                        "lat_max": LAT_MAX,
                        "similarity_threshold": similarity_threshold,
                        "chunk_size": chunk_size,
                        "sample_pixels_per_chunk": new_n,
                        "processed_chunk_indices": processed,
                        "buffer_features": df_points.to_dict("records"),
                    })
                    with open(CHECKPOINT_PATH, "wb") as f:
                        pickle.dump(checkpoint, f)
                    time.sleep(2)
                    continue

                print("❌ Memory limit persists. Marking chunk processed and skipping.")
                processed.add(idx)
                continue

            if ("quota" in msg) or ("rate" in msg) or ("too many" in msg):
                sleep_s = 30
                print(f"⚠️ Throttle/quota hit on chunk {idx}. Sleeping {sleep_s}s...")
                time.sleep(sleep_s)
                continue

            print("❌ Chunk failed:", e)
            continue

        feats = info.get("features", [])
        n_feats = len(feats)
        print(f"Chunk {idx} returned {n_feats} points")

        if n_feats:
            rows = []
            for f in feats:
                lon, lat = f["geometry"]["coordinates"]
                props = f.get("properties", {})
                rows.append([
                    float(lat),
                    float(lon),
                    int(props.get("habitat_index", -1)),
                    float(props.get("similarity", 0.0)),
                    int(idx),
                ])
            append_rows_to_csv(rows)

        processed.add(idx)
        checkpoint.update({
            "lon_min": LON_MIN,
            "lat_min": LAT_MIN,
            "lon_max": LON_MAX,
            "lat_max": LAT_MAX,
            "similarity_threshold": similarity_threshold,
            "chunk_size": chunk_size,
            "sample_pixels_per_chunk": sample_pixels_per_chunk,
            "processed_chunk_indices": processed,
            "buffer_features": df_points.to_dict("records"),
        })
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump(checkpoint, f)

        print(f"✅ DONE chunk {idx} in {(time.time()-t0)/60:.2f} min | checkpoint saved")
        time.sleep(random.uniform(0.5, 1.5))
        gc.collect()

    print("\n✅ All chunks processed.")
    print("✅ CSV saved at:", CSV_PATH)

# =========================
# 10. Main flow
# =========================
def run_kenya_similarity():
    if os.path.exists(CHECKPOINT_PATH):
        print("Checking for checkpoint...")
        print("Found checkpoint:", CHECKPOINT_PATH)

        with open(CHECKPOINT_PATH, "rb") as f:
            checkpoint = pickle.load(f)

        checkpoint = ensure_checkpoint_has_buffers(checkpoint)

        df_points = pd.DataFrame(checkpoint["buffer_features"])

        checkpoint["similarity_threshold"] = SIMILARITY_THRESHOLD
        checkpoint["chunk_size"] = CHUNK_SIZE
        checkpoint["sample_pixels_per_chunk"] = SAMPLE_PIXELS_PER_CHUNK

        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump(checkpoint, f)

        process_kenya_similarity_headless(
            df_points=df_points,
            similarity_threshold=SIMILARITY_THRESHOLD,
            chunk_size=CHUNK_SIZE,
            sample_pixels_per_chunk=SAMPLE_PIXELS_PER_CHUNK,
            scale=SCALE,
        )
        return

    # fresh run
    if os.path.exists(CSV_PATH):
        os.remove(CSV_PATH)

    initialize_csv_local([
        "Latitude", "Longitude", "Habitat ID", "Similarity Score", "Chunk Index"
    ])

    df_points = load_all_points()

    print("Total cleaned points:", len(df_points))
    print(df_points["label"].value_counts(dropna=False))
    print(df_points.head())

    # Save initial checkpoint
    checkpoint = {
        "lon_min": float(LON_MIN),
        "lat_min": float(LAT_MIN),
        "lon_max": float(LON_MAX),
        "lat_max": float(LAT_MAX),
        "similarity_threshold": float(SIMILARITY_THRESHOLD),
        "chunk_size": float(CHUNK_SIZE),
        "sample_pixels_per_chunk": int(SAMPLE_PIXELS_PER_CHUNK),
        "processed_chunk_indices": set(),
        "buffer_features": df_points.to_dict("records"),
    }
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(checkpoint, f)

    print("✅ Initial checkpoint saved:", CHECKPOINT_PATH)

    process_kenya_similarity_headless(
        df_points=df_points,
        similarity_threshold=SIMILARITY_THRESHOLD,
        chunk_size=CHUNK_SIZE,
        sample_pixels_per_chunk=SAMPLE_PIXELS_PER_CHUNK,
        scale=SCALE,
    )

# ---- Run ----
run_kenya_similarity()

✅ Earth Engine initialized
BASE_DIR: /Users/rs/Downloads/KENYA_HABITAT_SIMILARITY
CSV_PATH: /Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/Kenya_Habitat_Similarity.csv
CHECKPOINT_PATH: /Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/checkpoint.pkl
✅ Using Kenya boundary
Bounds: (33.910179252727914, -4.724105426383415, 41.91011868242448, 4.619998829490907)
Sentinel-2 image count after masking: 9448
Profile bands: ['ndwi', 'ndvi', 'mndwi', 'B2', 'B3', 'B4', 'B8', 'B11']
Total cleaned points: 106
label
1    55
0    51
Name: count, dtype: int64
  site_id  Longitude  Latitude       Date  label
0       1  34.782950 -0.092041 2024-12-01      1
1       2  34.789412 -0.077917 2024-01-01      1
2       3  34.781269 -0.095690 2024-12-01      1
3       4  34.785890 -0.095064 2024-12-01      1
4       5  34.816310 -0.077295 2024-12-01      1
✅ Initial checkpoint saved: /Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/checkpoint.pkl
Building profiles from buffered habitats: 106
Built profile 10/106
Built pro

In [12]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap

# =========================
# LOAD OUTPUT CSV
# =========================
CSV_TO_PLOT = "/Users/rs/Downloads/Kenya_Habitat_Similarity.csv"
# If your file is actually named si.csv, use this instead:
# CSV_TO_PLOT = "/Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/si.csv"

df_map = pd.read_csv(CSV_TO_PLOT)

print("Rows in similarity CSV:", len(df_map))
print(df_map.head())

# Drop bad rows if any
df_map = df_map.dropna(subset=["Latitude", "Longitude", "Similarity Score"]).copy()

# =========================
# CREATE BASE MAP
# =========================
center_lat = df_map["Latitude"].mean()
center_lon = df_map["Longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

# =========================
# ADD HEATMAP
# =========================
heat_data = df_map[["Latitude", "Longitude", "Similarity Score"]].values.tolist()

HeatMap(
    heat_data,
    radius=12,
    blur=10,
    max_zoom=8,
).add_to(m)

# =========================
# ADD COLORED POINTS
# =========================
marker_cluster = MarkerCluster().add_to(m)

def get_color(score):
    if score >= 97:
        return "darkgreen"
    elif score >= 94:
        return "green"
    elif score >= 91:
        return "orange"
    else:
        return "red"

for _, row in df_map.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=4,
        color=get_color(row["Similarity Score"]),
        fill=True,
        fill_opacity=0.8,
        popup=(
            f"Similarity: {row['Similarity Score']:.2f}<br>"
            f"Habitat ID: {row['Habitat ID']}<br>"
            f"Chunk: {row['Chunk Index']}"
        )
    ).add_to(marker_cluster)

# =========================
# SAVE + DISPLAY
# =========================
map_path = "/Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/kenya_similarity_map.html"
m.save(map_path)

print("✅ Map saved to:", map_path)
m

Rows in similarity CSV: 136
   Latitude  Longitude  Habitat ID  Similarity Score  Chunk Index
0  1.361349  34.850197           5         88.115993            6
1  1.299574  34.840874           6         89.607199            6
2  1.317281  34.852382          38         90.900694            6
3  1.401535  34.785656          10         91.706611            6
4  1.304955  34.855945          47         91.359759            6
✅ Map saved to: /Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/kenya_similarity_map.html


In [14]:
import pandas as pd
import ee
import geemap

# ============================================================
# 1. EARTH ENGINE INIT
# ============================================================
PROJECT_ID = "interpolated"

try:
    ee.Initialize(project=PROJECT_ID)
    print("✅ Earth Engine initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("✅ Earth Engine authenticated and initialized")

# ============================================================
# 2. FILE PATHS
# ============================================================
SIMILARITY_CSV = "/Users/rs/Downloads/Kenya_Habitat_Similarity.csv"
# If your file is actually named si.csv, use this instead:
# SIMILARITY_CSV = "/Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/si.csv"

FILE_1 = "/Users/rs/Downloads/positives & negatives kenya.xlsx"
FILE_2 = "/Users/rs/Downloads/Kenya.xlsx"

# ============================================================
# 3. HELPERS TO READ ORIGINAL POINTS
# ============================================================
def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_sheet(df, label):
    df = clean_columns(df)

    lon_candidates = [c for c in df.columns if c.lower().startswith("longitude")]
    lat_candidates = [c for c in df.columns if c.lower().startswith("latitude")]
    date_candidates = [c for c in df.columns if c.lower().startswith("date")]
    site_candidates = [c for c in df.columns if c.lower().startswith("site")]

    if not lon_candidates or not lat_candidates:
        raise ValueError(f"Could not find Longitude/Latitude columns in {list(df.columns)}")

    lon_col = lon_candidates[0]
    lat_col = lat_candidates[0]
    date_col = date_candidates[0] if date_candidates else None
    site_col = site_candidates[0] if site_candidates else None

    keep_cols = [lon_col, lat_col]
    if date_col:
        keep_cols.append(date_col)
    if site_col:
        keep_cols.append(site_col)

    out = df[keep_cols].copy()

    rename_map = {
        lon_col: "Longitude",
        lat_col: "Latitude",
    }
    if date_col:
        rename_map[date_col] = "Date"
    if site_col:
        rename_map[site_col] = "site_id"

    out = out.rename(columns=rename_map)

    if "Date" not in out.columns:
        out["Date"] = pd.NaT
    if "site_id" not in out.columns:
        out["site_id"] = range(1, len(out) + 1)

    out["Longitude"] = pd.to_numeric(out["Longitude"], errors="coerce")
    out["Latitude"] = pd.to_numeric(out["Latitude"], errors="coerce")
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["site_id"] = out["site_id"].astype(str)
    out["label"] = int(label)

    out = out.dropna(subset=["Longitude", "Latitude"]).copy()
    return out[["site_id", "Longitude", "Latitude", "Date", "label"]]

def load_all_points():
    pos1 = pd.read_excel(FILE_1, sheet_name="Positives")
    neg1 = pd.read_excel(FILE_1, sheet_name="Negatives")
    pos2 = pd.read_excel(FILE_2, sheet_name="Positive Sites")
    neg2 = pd.read_excel(FILE_2, sheet_name="Negatve Sites")

    pos1 = standardize_sheet(pos1, 1)
    neg1 = standardize_sheet(neg1, 0)
    pos2 = standardize_sheet(pos2, 1)
    neg2 = standardize_sheet(neg2, 0)

    df = pd.concat([pos1, neg1, pos2, neg2], ignore_index=True)
    df = df.drop_duplicates(subset=["Longitude", "Latitude", "label"]).reset_index(drop=True)
    return df

# ============================================================
# 4. LOAD DATA
# ============================================================
sim_df = pd.read_csv(SIMILARITY_CSV)
sim_df.columns = [str(c).strip() for c in sim_df.columns]
sim_df = sim_df.dropna(subset=["Latitude", "Longitude", "Similarity Score"]).copy()

known_df = load_all_points()

print("✅ Similarity points:", len(sim_df))
print("✅ Known points:", len(known_df))
print(known_df["label"].value_counts(dropna=False))

# ============================================================
# 5. KENYA BOUNDARY
# ============================================================
kenya_fc = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Kenya"))
)
roi = kenya_fc.geometry()

# ============================================================
# 6. DATAFRAME -> FEATURE COLLECTION
# ============================================================
def df_to_fc(df, lon_col="Longitude", lat_col="Latitude", prop_cols=None):
    prop_cols = prop_cols or []
    features = []

    for _, row in df.iterrows():
        props = {}
        for col in prop_cols:
            val = row[col]
            if pd.isna(val):
                props[col] = None
            else:
                if hasattr(val, "strftime"):
                    props[col] = val.strftime("%Y-%m-%d")
                else:
                    props[col] = val.item() if hasattr(val, "item") else val

        ft = ee.Feature(
            ee.Geometry.Point([float(row[lon_col]), float(row[lat_col])]),
            props
        )
        features.append(ft)

    return ee.FeatureCollection(features)

sim_fc = df_to_fc(
    sim_df,
    prop_cols=["Similarity Score", "Habitat ID", "Chunk Index"]
)

pos_fc = df_to_fc(
    known_df[known_df["label"] == 1].copy(),
    prop_cols=["site_id", "label"]
)

neg_fc = df_to_fc(
    known_df[known_df["label"] == 0].copy(),
    prop_cols=["site_id", "label"]
)

# ============================================================
# 7. STYLE SIMILARITY POINTS BY SCORE
# ============================================================
def style_similarity(feature):
    sim = ee.Number(feature.get("Similarity Score"))

    color = ee.Algorithms.If(
        sim.gte(97), "#006400",   # dark green
        ee.Algorithms.If(
            sim.gte(94), "#32CD32",   # lime green
            ee.Algorithms.If(
                sim.gte(91), "#FFA500",   # orange
                "#FF0000"   # red
            )
        )
    )

    return feature.set({
        "style": {
            "color": color,
            "fillColor": color,
            "width": 1,
            "pointSize": 4
        }
    })

styled_sim_fc = sim_fc.map(style_similarity)

# ============================================================
# 8. CREATE MAP
# ============================================================
center_lat = sim_df["Latitude"].mean()
center_lon = sim_df["Longitude"].mean()

Map = geemap.Map(center=[center_lat, center_lon], zoom=6)

# Basemap choices
Map.add_basemap("HYBRID")

# Kenya boundary
Map.addLayer(
    kenya_fc.style(**{
        "color": "black",
        "fillColor": "00000000",
        "width": 2
    }),
    {},
    "Kenya Boundary"
)

# Known positive sites
Map.addLayer(
    pos_fc.style(**{
        "color": "blue",
        "fillColor": "blue",
        "width": 1,
        "pointSize": 6
    }),
    {},
    "Known Positive Sites"
)

# Known negative sites
Map.addLayer(
    neg_fc.style(**{
        "color": "black",
        "fillColor": "black",
        "width": 1,
        "pointSize": 6
    }),
    {},
    "Known Negative Sites"
)

# Predicted similarity points
Map.addLayer(
    styled_sim_fc.style(**{"styleProperty": "style"}),
    {},
    "Predicted Similar Habitats"
)

# ============================================================
# 9. OPTIONAL LEGEND
# ============================================================
legend_dict = {
    "Similarity ≥ 97": "#006400",
    "Similarity 94–96.99": "#32CD32",
    "Similarity 91–93.99": "#FFA500",
    "Similarity 88–90.99": "#FF0000",
    "Known Positive Site": "blue",
    "Known Negative Site": "black",
}

Map.add_legend(title="Habitat Similarity", legend_dict=legend_dict)

# ============================================================
# 10. OPTIONAL CENTER ON KENYA
# ============================================================
Map.centerObject(kenya_fc, 6)

# ============================================================
# 11. DISPLAY MAP IN NOTEBOOK
# ============================================================
Map

✅ Earth Engine initialized
✅ Similarity points: 136
✅ Known points: 106
label
1    55
0    51
Name: count, dtype: int64


Map(center=[0.5336326464912944, 37.86026492779467], controls=(WidgetControl(options=['position', 'transparent_…

In [15]:
import pandas as pd
import geemap
import ee

# Load similarity CSV
CSV_PATH = "/Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/Kenya_Habitat_Similarity.csv"
# If needed use:
# CSV_PATH = "/Users/rs/Downloads/KENYA_HABITAT_SIMILARITY/si.csv"

df = pd.read_csv(CSV_PATH)

# Remove bad rows
df = df.dropna(subset=["Latitude", "Longitude"])

print("Points loaded:", len(df))

# Convert dataframe to Earth Engine FeatureCollection
features = []

for _, row in df.iterrows():
    geom = ee.Geometry.Point([row["Longitude"], row["Latitude"]])
    ft = ee.Feature(
        geom,
        {
            "similarity": row["Similarity Score"],
            "habitat": row["Habitat ID"],
            "chunk": row["Chunk Index"],
        },
    )
    features.append(ft)

fc = ee.FeatureCollection(features)

# Create interactive map
Map = geemap.Map(center=[0.5, 37], zoom=6)

# Add similarity points
Map.addLayer(
    fc,
    {"color": "red"},
    "Similarity Points"
)

# Show map in notebook
Map

Points loaded: 136


Map(center=[0.5, 37], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…